# V15 — CICIDS2017 Improved integration and normal/unknown split
Preserves `Dst Port` as threshold metadata, not as an Autoencoder input. Attempted flows are relabeled BENIGN before splitting.


In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'/'CICIDS2017_improved'
OUT = ROOT / 'data' / 'processed' / 'unknown_detection'
OUT.mkdir(parents=True,exist_ok=True)
SEED=42
# Intended attack families; exact labels are resolved against the loaded CSVs.
UNKNOWN_CLASSES=['Web Attack - XSS','FTP-Patator','DoS Slowhttptest']
print('Input:',RAW,'Output:',OUT)


Input: C:\Users\saaral\Desktop\gc\pattern recog\project\]code\Hybrid-Network-Intrusion-Detection\data\raw\CICIDS2017_improved Output: C:\Users\saaral\Desktop\gc\pattern recog\project\]code\Hybrid-Network-Intrusion-Detection\data\processed\unknown_detection


## Load all eight CSVs from data/raw

In [9]:
# Select only CSV files in data/raw; do not mix CICIDS2017_improved automatically.
files=sorted(RAW.glob('*.csv'))
if not files: raise FileNotFoundError(f'No CSV files in {RAW}')
frames=[]
for p in files:
    try: frame=pd.read_csv(p,low_memory=False,encoding='utf-8-sig')
    except UnicodeDecodeError: frame=pd.read_csv(p,low_memory=False,encoding='latin-1')
    frame.columns=frame.columns.str.strip()
    if 'Label' not in frame: raise ValueError(f'No Label column in {p.name}')
    frame['Label']=frame['Label'].astype(str).str.strip()
    frame['source_file']=p.name
    frames.append(frame)
    print(p.name,frame.shape)
data=pd.concat(frames,ignore_index=True)
del frames
print('Raw label counts:',data.Label.value_counts().to_string())


friday.csv (547557, 92)
monday.csv (371624, 92)
thursday.csv (362076, 92)
tuesday.csv (322078, 92)
wednesday.csv (496641, 92)
Raw label counts: Label
BENIGN                                    1582566
Portscan                                   159066
DoS Hulk                                   158468
DDoS                                        95144
Infiltration - Portscan                     71767
DoS GoldenEye                                7567
Botnet - Attempted                           4067
FTP-Patator                                  3972
DoS Slowloris                                3859
DoS Slowhttptest - Attempted                 3368
SSH-Patator                                  2961
DoS Slowloris - Attempted                    1847
DoS Slowhttptest                             1740
Web Attack - Brute Force - Attempted         1292
Botnet                                        736
Web Attack - XSS - Attempted                  655
DoS Hulk - Attempted                          581


## Clean, resolve conflicts, remove duplicates and redundant columns

In [11]:
# Preserve original notebook policy, adapted to either classic or improved CIC columns.
label_map={'Web Attack � Brute Force':'Web Attack - Brute Force','Web Attack � Sql Injection':'Web Attack - SQL Injection','Web Attack � XSS':'Web Attack - XSS'}
data['Label']=data['Label'].replace(label_map)
# Normalize alternate dash and replacement-character variants in XSS labels.
data['Label']=data['Label'].str.replace(r'^Web Attack\s*[-–—�]\s*XSS(\s*-\s*Attempted)?$', lambda m: 'Web Attack - XSS' + (' - Attempted' if m.group(1) else ''), regex=True)
# Dataset-author policy: attempted flows are non-malicious, relabel as BENIGN.
# Keep an audit count, but do not feed attempted-category into the model.
if 'Attempted Category' in data.columns:
    attempted_flag = pd.to_numeric(data['Attempted Category'], errors='coerce')
    attempted_mask = attempted_flag.notna() & attempted_flag.ne(-1)
else:
    attempted_mask = pd.Series(False, index=data.index)
attempted_mask |= data['Label'].str.contains(r'\s*-\s*Attempted$', case=False, na=False)
print('Attempted flows relabeled BENIGN:', int(attempted_mask.sum()))
data.loc[attempted_mask, 'Label'] = 'BENIGN'

# Drop identifying/leakage fields when present, keep source_file as metadata.
identifiers=['Flow ID','Src IP','Dst IP','Src Port','Timestamp','Attempted Category','id']
data=data.drop(columns=identifiers,errors='ignore')
for c in data.columns.difference(['Label','source_file']):
    data[c]=pd.to_numeric(data[c],errors='coerce')
features=[c for c in data if c not in ['Label','source_file','Dst Port']]
data[features]=data[features].replace([np.inf,-np.inf],np.nan)
invalid=data[features].isna().any(axis=1)
print('Missing/Inf rows:',int(invalid.sum()))
data=data.loc[~invalid].copy()
for c in ['Flow Duration','Flow Bytes/s','Flow Packets/s']:
    if c in data: data=data.loc[data[c]>=0].copy()
# Do not drop all negative measurements: IAT and initial-window sentinel negatives may be legitimate.
for c in ['Fwd Header Length','Bwd Header Length','min_seg_size_forward','Fwd Seg Size Min']:
    if c in data: data=data.loc[data[c]>=0].copy()
# Remove exact feature conflicts and duplicates; port remains metadata.

features=[c for c in data if c not in ['Label','source_file','Dst Port']]
h=pd.util.hash_pandas_object(data[features],index=False)
conflict=data.groupby(h,sort=False)['Label'].nunique()
data=data.loc[~h.isin(conflict[conflict>1].index)].copy()
h=pd.util.hash_pandas_object(data[features],index=False)
data=data.loc[~h.duplicated()].copy()
# Remove explicitly redundant columns, then all constant numeric columns.
data=data.drop(columns=['Fwd Header Length.1','Bwd PSH Flags','Bwd URG Flags','Fwd Avg Bytes/Bulk','Fwd Avg Packets/Bulk','Fwd Avg Bulk Rate','Bwd Avg Bytes/Bulk','Bwd Avg Packets/Bulk','Bwd Avg Bulk Rate','Fwd Bytes/Bulk Avg','Fwd Packet/Bulk Avg','Fwd Bulk Rate Avg','Bwd Bytes/Bulk Avg','Bwd Packet/Bulk Avg','Bwd Bulk Rate Avg'],errors='ignore')
features=[c for c in data if c not in ['Label','source_file','Dst Port']]
constants=[c for c in features if data[c].nunique(dropna=False)<=1]
data=data.drop(columns=constants).reset_index(drop=True)
print('Clean:',data.shape,'Labels:',data.Label.value_counts().to_dict(),'Constant removed:',constants)


Attempted flows relabeled BENIGN: 0
Missing/Inf rows: 0
Clean: (1714507, 77) Labels: {'BENIGN': 1432319, 'DoS Hulk': 158468, 'DDoS': 95144, 'DoS GoldenEye': 7567, 'Infiltration - Portscan': 6026, 'FTP-Patator': 3972, 'DoS Slowloris': 3859, 'SSH-Patator': 2961, 'DoS Slowhttptest': 1740, 'Portscan': 1572, 'Botnet': 736, 'Web Attack - Brute Force': 73, 'Infiltration': 31, 'Web Attack - XSS': 18, 'Web Attack - SQL Injection': 13, 'Heartbleed': 8} Constant removed: []


## Split BENIGN into train/validation/test (70/15/15), then create a balanced held-out unknown test
Unknown classes never enter training or validation. The balanced test uses only a subset of normal_test, with no sample reuse between the three attack-specific tests.

In [12]:
# Resolve exact labels present in the improved CICIDS2017 files.
# Do not silently include " - Attempted" variants.
available = sorted(data['Label'].dropna().astype(str).unique())
print('Available labels:', available)

def resolve_attack_label(requested, available_labels):
    aliases = {
        'Web Attack - XSS': ['Web Attack - XSS', 'Web Attack – XSS', 'Web Attack � XSS'],
        'FTP-Patator': ['FTP-Patator', 'FTP Patator', 'FTP-BruteForce', 'FTP-Brute Force'],
        'DoS Slowhttptest': ['DoS Slowhttptest', 'DoS SlowHTTPTest', 'DoS Slow HTTP Test'],
    }
    for candidate in aliases.get(requested, [requested]):
        matches = [label for label in available_labels if label.casefold() == candidate.casefold()]
        if len(matches) == 1:
            return matches[0]
    raise ValueError(f'Cannot resolve {requested!r}. Available labels: {available_labels}')

UNKNOWN_CLASSES = [resolve_attack_label(x, available) for x in UNKNOWN_CLASSES]
print('Resolved unknown classes:', UNKNOWN_CLASSES)
normal = data.loc[data.Label.eq('BENIGN')].copy()
unknown = data.loc[data.Label.isin(UNKNOWN_CLASSES)].copy()
if len(normal) < 7:
    raise ValueError('Not enough BENIGN rows to split into train/validation/test')
normal_train, normal_holdout = train_test_split(normal, test_size=0.30, random_state=SEED, shuffle=True)
normal_validation, normal_test = train_test_split(normal_holdout, test_size=0.50, random_state=SEED, shuffle=True)
normal_pool = normal_test.sample(frac=1, random_state=SEED).reset_index(drop=True)
attack_counts = {attack: int(unknown.Label.eq(attack).sum()) for attack in UNKNOWN_CLASSES}
allocation = {attack: 0 for attack in UNKNOWN_CLASSES}
remaining = len(normal_pool)
while remaining:
    eligible = [attack for attack in UNKNOWN_CLASSES if allocation[attack] < attack_counts[attack]]
    if not eligible:
        break
    for attack in eligible:
        if remaining == 0:
            break
        allocation[attack] += 1
        remaining -= 1
if any(n == 0 for n in allocation.values()):
    raise ValueError('Insufficient held-out BENIGN rows for each unknown attack')
attack_test_sets = {}
position = 0
for attack in UNKNOWN_CLASSES:
    n = allocation[attack]
    normal_part = normal_pool.iloc[position:position+n]
    position += n
    attack_part = unknown.loc[unknown.Label.eq(attack)].sample(n=n, random_state=SEED)
    combined = pd.concat([normal_part, attack_part], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    assert combined.Label.eq('BENIGN').sum() == combined.Label.eq(attack).sum() == n
    attack_test_sets[attack] = combined
    print(f'{attack}: {n} BENIGN + {n} unknown')
balanced_unknown_test = pd.concat(attack_test_sets.values(), ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
assert balanced_unknown_test.Label.eq('BENIGN').sum() == balanced_unknown_test.Label.ne('BENIGN').sum()
assert all(x.Label.eq('BENIGN').all() for x in (normal_train, normal_validation, normal_test))
print('Normal train/validation/test:', len(normal_train), len(normal_validation), len(normal_test))
print('All unknown test:', len(unknown))
print('Balanced unknown test:', balanced_unknown_test.Label.value_counts().to_dict())


Available labels: ['BENIGN', 'Botnet', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS Slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'Infiltration - Portscan', 'Portscan', 'SSH-Patator', 'Web Attack - Brute Force', 'Web Attack - SQL Injection', 'Web Attack - XSS']
Resolved unknown classes: ['Web Attack - XSS', 'FTP-Patator', 'DoS Slowhttptest']
Web Attack - XSS: 18 BENIGN + 18 unknown
FTP-Patator: 3972 BENIGN + 3972 unknown
DoS Slowhttptest: 1740 BENIGN + 1740 unknown
Normal train/validation/test: 1002623 214848 214848
All unknown test: 5730
Balanced unknown test: {'BENIGN': 5730, 'FTP-Patator': 3972, 'DoS Slowhttptest': 1740, 'Web Attack - XSS': 18}


## Save only normal training, normal validation, normal testing, and unknown testing datasets under `data/processed/unknown_detection/`

In [13]:
outputs={
    'normal_train':normal_train,
    'normal_validation':normal_validation,
    'normal_test':normal_test,
    'unknown_test':unknown,
    'balanced_unknown_test':balanced_unknown_test,
}
for attack,frame in attack_test_sets.items():
    slug=''.join(c.lower() if c.isalnum() else '_' for c in attack).strip('_')
    outputs[f'balanced_{slug}_test']=frame
for name,frame in outputs.items():
    frame.to_csv(OUT/f'{name}.csv',index=False)
    print(f'Saved {name}.csv: {len(frame)} rows')
pd.DataFrame({'feature':[c for c in data if c not in ['Label','source_file','Dst Port']]}).to_csv(OUT/'feature_list.csv',index=False)
pd.DataFrame([{'dataset':name,'rows':len(frame),'benign':int(frame.Label.eq('BENIGN').sum()),'unknown':int(frame.Label.ne('BENIGN').sum())} for name,frame in outputs.items()]).to_csv(OUT/'manifest.csv',index=False)

Saved normal_train.csv: 1002623 rows
Saved normal_validation.csv: 214848 rows
Saved normal_test.csv: 214848 rows
Saved unknown_test.csv: 5730 rows
Saved balanced_unknown_test.csv: 11460 rows
Saved balanced_web_attack___xss_test.csv: 36 rows
Saved balanced_ftp_patator_test.csv: 7944 rows
Saved balanced_dos_slowhttptest_test.csv: 3480 rows
